# AtomicVision Judge Repro Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adityabaskati-weeb/-AtomicVision-An-Autonomous-AI-Agent-for-Non-Destructive-Multi-Defect-Mapping/blob/main/notebooks/AtomicVision_Judge_Repro_Colab.ipynb)

This is the shareable, reproducible judge notebook for AtomicVision.

It now documents three reproducible tracks side by side:

1. **Track A: Full rebuild** from the official `two_step_curriculum` dataset into the stable merged adapter.
2. **Track B: Targeted post-recovery booster** that restores the saved good adapter from Hugging Face Hub, continues SFT with `--init-adapter-dir`, and re-runs held-out evaluation.
3. **Track C: Hard-frontier booster** that continues from the promoted medium-fidelity boost adapter and specifically targets the remaining hard-case gap.

Permanent seed policy: SFT rebuild and continuation data stay in the `1000-3999` band, GRPO prompt selection stays in the `4000-7999` band, and held-out evaluation uses `10000-10999` only.

The current best published checkpoint is `prodigyhuh/atomicvision-medium-fidelity-boost-lora`. The stable fallback adapter is `prodigyhuh/atomicvision-format-submit-merged-lora`, and Track C is the next frontier-focused continuation path if you want to keep improving the hard slice.

Recommended runtime: Colab T4 or better.

Optional secret: `HF_TOKEN` in Colab secrets if you want automatic adapter upload.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Adityabaskati-weeb/-AtomicVision-An-Autonomous-AI-Agent-for-Non-Destructive-Multi-Defect-Mapping.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/AtomicVision")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())
print("branch:", BRANCH)


In [ ]:
import subprocess
import sys
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements-grpo.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openenv-core==0.2.3", "huggingface_hub"], check=True)

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets. Hub publish will be skipped.")
except Exception:
    print("google.colab.userdata unavailable. Hub publish will be skipped unless HF_TOKEN is already set.")


In [ ]:
import json
from pathlib import Path

RUN_REBUILD_TRACK = True
RUN_BOOST_TRACK = True
RUN_HARD_FRONTIER_TRACK = False

REBUILD_SFT_SEED_START = 1000
BOOST_SFT_SEED_START = 2000
HARD_FRONTIER_SFT_SEED_START = 3000
HELDOUT_EVAL_SEED_START = 10000

DATASET = PROJECT_DIR / "outputs" / "sft" / "atomicvision_two_step_curriculum_sft.jsonl"
VALIDATE_OUT = Path("/content/atomicvision-two-step-validate")
SANITY_OUT = Path("/content/atomicvision-format-submit-merged-sanity-lora")
FINAL_OUT = Path("/content/atomicvision-format-submit-merged-lora")
EVAL_JSON = PROJECT_DIR / "outputs" / "evaluation" / "atomicvision_judge_colab_eval.json"
HF_REPO_ID = "prodigyhuh/atomicvision-format-submit-merged-lora"

BASE_ADAPTER_REPO_ID = "prodigyhuh/atomicvision-format-submit-merged-lora"
BASE_ADAPTER_DIR = Path("/content/atomicvision-format-submit-merged-lora-base")
BOOST_DATASET = PROJECT_DIR / "outputs" / "sft" / "atomicvision_medium_prior_fidelity_sft.jsonl"
BOOST_VALIDATE_OUT = Path("/content/atomicvision-medium-fidelity-validate")
BOOST_OUT = Path("/content/atomicvision-medium-fidelity-boost-lora")
BOOST_EVAL_JSON = PROJECT_DIR / "outputs" / "evaluation" / "atomicvision_medium_fidelity_boost_eval.json"
BOOST_HF_REPO_ID = "prodigyhuh/atomicvision-medium-fidelity-boost-lora"

HARD_FRONTIER_BASE_REPO_ID = "prodigyhuh/atomicvision-medium-fidelity-boost-lora"
HARD_FRONTIER_BASE_DIR = Path("/content/atomicvision-medium-fidelity-boost-base")
HARD_FRONTIER_DATASET = PROJECT_DIR / "outputs" / "sft" / "atomicvision_hard_frontier_boost_sft.jsonl"
HARD_FRONTIER_VALIDATE_OUT = Path("/content/atomicvision-hard-frontier-validate")
HARD_FRONTIER_OUT = Path("/content/atomicvision-hard-frontier-boost-lora")
HARD_FRONTIER_EVAL_JSON = PROJECT_DIR / "outputs" / "evaluation" / "atomicvision_hard_frontier_boost_eval.json"
HARD_FRONTIER_HF_REPO_ID = "prodigyhuh/atomicvision-hard-frontier-boost-lora"

print("RUN_REBUILD_TRACK =", RUN_REBUILD_TRACK)
print("RUN_BOOST_TRACK =", RUN_BOOST_TRACK)
print("RUN_HARD_FRONTIER_TRACK =", RUN_HARD_FRONTIER_TRACK)
print("rebuild dataset:", DATASET)
print("rebuild adapter:", FINAL_OUT)
print("boost dataset:", BOOST_DATASET)
print("boost adapter:", BOOST_OUT)
print("hard frontier dataset:", HARD_FRONTIER_DATASET)
print("hard frontier adapter:", HARD_FRONTIER_OUT)
print("rebuild SFT seed start:", REBUILD_SFT_SEED_START)
print("boost SFT seed start:", BOOST_SFT_SEED_START)
print("hard frontier SFT seed start:", HARD_FRONTIER_SFT_SEED_START)
print("held-out eval seed start:", HELDOUT_EVAL_SEED_START)


def print_eval_table(report_path: Path, title: str) -> None:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(title)
    print("| difficulty | policy | reward | outcome | penalty | f1 | mae | steps | cost | fail | done | strict | normalized | first_valid | first_prior | submit |")
    print("| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |")
    for difficulty, result in report["results"].items():
        for key, label in [
            ("baseline_prior_submit", "baseline"),
            ("strict_adapter", "strict_adapter"),
            ("normalized_adapter", "normalized_adapter"),
        ]:
            row = result[key]
            print(
                f"| {difficulty} | {label} | "
                f"{row['mean_reward']:.4f} | {row['mean_outcome_reward_total']:.4f} | {row['mean_penalty_total']:.4f} | "
                f"{row['mean_f1']:.4f} | {row['mean_mae']:.5f} | "
                f"{row['mean_steps']:.2f} | {row['mean_scan_cost']:.2f} | "
                f"{row['tool_failure_rate']:.3f} | {row['done_rate']:.3f} | "
                f"{row['strict_tool_call_pass_rate']:.2f} | {row['normalized_tool_call_pass_rate']:.2f} | "
                f"{row['first_action_valid_rate']:.2f} | {row['first_action_ask_prior_rate']:.2f} | "
                f"{row['submit_action_rate']:.2f} |"
            )
    print("
Saved JSON:", report_path)


## Track A: Full Rebuild

This is the conservative end-to-end path: regenerate the official `two_step_curriculum` dataset, validate it with the NaN-safe trainer, rebuild the merged adapter, publish it, and run strict + normalized held-out evaluation.


In [ ]:
import subprocess
import sys

if RUN_REBUILD_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/generate_atomicvision_sft_data.py",
            "--profile", "two_step_curriculum",
            "--episodes-per-difficulty", "256",
            "--seed-start", str(REBUILD_SFT_SEED_START),
            "--difficulties", "medium", "hard",
            "--min-scan-improvement", "0.25",
            "--output-jsonl", str(DATASET),
        ],
        check=True,
    )
else:
    print("Skipping Track A dataset generation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_REBUILD_TRACK:
    if VALIDATE_OUT.exists():
        shutil.rmtree(VALIDATE_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(DATASET),
            "--output-dir", str(VALIDATE_OUT),
            "--validate-only",
        ],
        check=True,
    )
else:
    print("Skipping Track A validation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_REBUILD_TRACK:
    if SANITY_OUT.exists():
        shutil.rmtree(SANITY_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(DATASET),
            "--model", "Qwen/Qwen3-1.7B",
            "--output-dir", str(SANITY_OUT),
            "--max-examples", "64",
            "--max-updates", "5",
            "--grad-accum", "8",
            "--max-length", "1536",
            "--learning-rate", "1e-5",
            "--max-grad-norm", "1.0",
            "--checkpoint-steps", "5",
            "--overwrite-output-dir",
        ],
        check=True,
    )
else:
    print("Skipping Track A sanity train.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_REBUILD_TRACK:
    if FINAL_OUT.exists():
        shutil.rmtree(FINAL_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(DATASET),
            "--model", "Qwen/Qwen3-1.7B",
            "--output-dir", str(FINAL_OUT),
            "--max-updates", "40",
            "--grad-accum", "8",
            "--max-length", "1536",
            "--learning-rate", "1e-5",
            "--max-grad-norm", "1.0",
            "--checkpoint-steps", "20", "40",
            "--overwrite-output-dir",
        ],
        check=True,
    )
    print("checkpoint-20:", FINAL_OUT / "checkpoint-20")
    print("checkpoint-40:", FINAL_OUT / "checkpoint-40")
else:
    print("Skipping Track A rebuild.")


In [ ]:
import os
import subprocess
import sys

if RUN_REBUILD_TRACK and os.environ.get("HF_TOKEN"):
    subprocess.run(
        [
            sys.executable,
            "training/publish_adapter_to_hub.py",
            "--adapter-dir", str(FINAL_OUT),
            "--repo-id", HF_REPO_ID,
            "--base-model", "Qwen/Qwen3-1.7B",
            "--include-zip",
        ],
        check=True,
    )
    print("HF publish done:", f"https://huggingface.co/{HF_REPO_ID}")
elif RUN_REBUILD_TRACK:
    print("HF_TOKEN not set, so Track A publish was skipped.")
else:
    print("Skipping Track A publish.")


In [ ]:
import subprocess
import sys

if RUN_REBUILD_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/evaluate_atomicvision_adapter.py",
            "--adapter-dir", str(FINAL_OUT),
            "--base-model", "Qwen/Qwen3-1.7B",
            "--difficulties", "medium", "hard",
            "--episodes", "32",
            "--seed-start", str(HELDOUT_EVAL_SEED_START),
            "--modes", "strict", "normalized",
            "--output-json", str(EVAL_JSON),
        ],
        check=True,
    )
else:
    print("Skipping Track A eval.")


In [ ]:
if RUN_REBUILD_TRACK:
    print_eval_table(EVAL_JSON, "TRACK A HELD-OUT EVAL (HONEST GATE)")
else:
    print("Skipping Track A table render.")


## Track B: Targeted Post-Recovery Booster

This is the smaller paper-backed continuation pass. It restores the current good adapter from Hugging Face Hub, builds a medium-only high-`submit_prior` dataset, continues SFT with `--init-adapter-dir`, and then re-runs the same held-out evaluator.


In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

if RUN_BOOST_TRACK:
    if BASE_ADAPTER_DIR.exists():
        shutil.rmtree(BASE_ADAPTER_DIR)

    snapshot_download(
        repo_id=BASE_ADAPTER_REPO_ID,
        repo_type="model",
        local_dir=str(BASE_ADAPTER_DIR),
        local_dir_use_symlinks=False,
        token=os.environ.get("HF_TOKEN"),
        ignore_patterns=["*.zip"],
    )
    print("base adapter restored:", BASE_ADAPTER_DIR)
else:
    print("Skipping Track B adapter restore.")


In [ ]:
import subprocess
import sys

if RUN_BOOST_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/generate_atomicvision_sft_data.py",
            "--profile", "cost_aware",
            "--episodes-per-difficulty", "256",
            "--seed-start", str(BOOST_SFT_SEED_START),
            "--difficulties", "medium",
            "--submit-prior-ratio", "0.95",
            "--reference-ratio", "0.02",
            "--min-scan-improvement", "0.25",
            "--output-jsonl", str(BOOST_DATASET),
        ],
        check=True,
    )
else:
    print("Skipping Track B dataset generation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_BOOST_TRACK:
    if BOOST_VALIDATE_OUT.exists():
        shutil.rmtree(BOOST_VALIDATE_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(BOOST_DATASET),
            "--output-dir", str(BOOST_VALIDATE_OUT),
            "--validate-only",
        ],
        check=True,
    )
else:
    print("Skipping Track B validation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_BOOST_TRACK:
    if BOOST_OUT.exists():
        shutil.rmtree(BOOST_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(BOOST_DATASET),
            "--model", "Qwen/Qwen3-1.7B",
            "--init-adapter-dir", str(BASE_ADAPTER_DIR),
            "--output-dir", str(BOOST_OUT),
            "--max-updates", "12",
            "--grad-accum", "8",
            "--batch-size", "1",
            "--max-length", "1536",
            "--learning-rate", "5e-6",
            "--max-grad-norm", "1.0",
            "--checkpoint-steps", "6", "12",
            "--overwrite-output-dir",
        ],
        check=True,
    )
    print("checkpoint-6:", BOOST_OUT / "checkpoint-6")
    print("checkpoint-12:", BOOST_OUT / "checkpoint-12")
else:
    print("Skipping Track B continuation.")


In [ ]:
import subprocess
import sys

if RUN_BOOST_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/evaluate_atomicvision_adapter.py",
            "--adapter-dir", str(BOOST_OUT),
            "--base-model", "Qwen/Qwen3-1.7B",
            "--difficulties", "medium", "hard",
            "--episodes", "32",
            "--seed-start", str(HELDOUT_EVAL_SEED_START),
            "--modes", "strict", "normalized",
            "--output-json", str(BOOST_EVAL_JSON),
        ],
        check=True,
    )
else:
    print("Skipping Track B eval.")


In [ ]:
if RUN_BOOST_TRACK:
    print_eval_table(BOOST_EVAL_JSON, "TRACK B HELD-OUT EVAL (TARGETED BOOST)")
else:
    print("Skipping Track B table render.")


In [ ]:
import os
import subprocess
import sys

if RUN_BOOST_TRACK and os.environ.get("HF_TOKEN"):
    subprocess.run(
        [
            sys.executable,
            "training/publish_adapter_to_hub.py",
            "--adapter-dir", str(BOOST_OUT),
            "--repo-id", BOOST_HF_REPO_ID,
            "--base-model", "Qwen/Qwen3-1.7B",
            "--include-zip",
        ],
        check=True,
    )
    print("HF publish done:", f"https://huggingface.co/{BOOST_HF_REPO_ID}")
elif RUN_BOOST_TRACK:
    print("HF_TOKEN not set, so Track B publish was skipped.")
else:
    print("Skipping Track B publish.")


## Track C: Hard-Frontier Booster

This is the next targeted continuation pass. It restores the promoted medium-fidelity boost adapter from Hugging Face Hub, builds a `hard_frontier_boost` dataset with a wider reference-search budget, continues SFT with `--init-adapter-dir`, and re-runs strict + normalized held-out evaluation on the hard slice focus.


In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

if RUN_HARD_FRONTIER_TRACK:
    if HARD_FRONTIER_BASE_DIR.exists():
        shutil.rmtree(HARD_FRONTIER_BASE_DIR)

    snapshot_download(
        repo_id=HARD_FRONTIER_BASE_REPO_ID,
        repo_type="model",
        local_dir=str(HARD_FRONTIER_BASE_DIR),
        local_dir_use_symlinks=False,
        token=os.environ.get("HF_TOKEN"),
        ignore_patterns=["*.zip"],
    )
    print("hard-frontier base adapter restored:", HARD_FRONTIER_BASE_DIR)
else:
    print("Skipping Track C adapter restore.")


In [ ]:
import subprocess
import sys

if RUN_HARD_FRONTIER_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/generate_atomicvision_sft_data.py",
            "--profile", "hard_frontier_boost",
            "--episodes-per-difficulty", "256",
            "--seed-start", str(HARD_FRONTIER_SFT_SEED_START),
            "--difficulties", "hard",
            "--max-scan-candidates-per-difficulty", "1024",
            "--output-jsonl", str(HARD_FRONTIER_DATASET),
        ],
        check=True,
    )
else:
    print("Skipping Track C dataset generation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_HARD_FRONTIER_TRACK:
    if HARD_FRONTIER_VALIDATE_OUT.exists():
        shutil.rmtree(HARD_FRONTIER_VALIDATE_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(HARD_FRONTIER_DATASET),
            "--output-dir", str(HARD_FRONTIER_VALIDATE_OUT),
            "--validate-only",
        ],
        check=True,
    )
else:
    print("Skipping Track C validation.")


In [ ]:
import shutil
import subprocess
import sys

if RUN_HARD_FRONTIER_TRACK:
    if HARD_FRONTIER_OUT.exists():
        shutil.rmtree(HARD_FRONTIER_OUT)

    subprocess.run(
        [
            sys.executable,
            "training/train_sft_atomicvision_safe.py",
            "--dataset-jsonl", str(HARD_FRONTIER_DATASET),
            "--model", "Qwen/Qwen3-1.7B",
            "--init-adapter-dir", str(HARD_FRONTIER_BASE_DIR),
            "--output-dir", str(HARD_FRONTIER_OUT),
            "--max-updates", "12",
            "--grad-accum", "8",
            "--batch-size", "1",
            "--max-length", "1536",
            "--learning-rate", "5e-6",
            "--max-grad-norm", "1.0",
            "--checkpoint-steps", "6", "12",
            "--overwrite-output-dir",
        ],
        check=True,
    )
    print("checkpoint-6:", HARD_FRONTIER_OUT / "checkpoint-6")
    print("checkpoint-12:", HARD_FRONTIER_OUT / "checkpoint-12")
else:
    print("Skipping Track C continuation.")


In [ ]:
import subprocess
import sys

if RUN_HARD_FRONTIER_TRACK:
    subprocess.run(
        [
            sys.executable,
            "training/evaluate_atomicvision_adapter.py",
            "--adapter-dir", str(HARD_FRONTIER_OUT),
            "--base-model", "Qwen/Qwen3-1.7B",
            "--difficulties", "medium", "hard",
            "--episodes", "32",
            "--seed-start", str(HELDOUT_EVAL_SEED_START),
            "--modes", "strict", "normalized",
            "--output-json", str(HARD_FRONTIER_EVAL_JSON),
        ],
        check=True,
    )
else:
    print("Skipping Track C eval.")


In [ ]:
if RUN_HARD_FRONTIER_TRACK:
    print_eval_table(HARD_FRONTIER_EVAL_JSON, "TRACK C HELD-OUT EVAL (HARD FRONTIER BOOST)")
else:
    print("Skipping Track C table render.")


In [ ]:
import os
import subprocess
import sys

if RUN_HARD_FRONTIER_TRACK and os.environ.get("HF_TOKEN"):
    subprocess.run(
        [
            sys.executable,
            "training/publish_adapter_to_hub.py",
            "--adapter-dir", str(HARD_FRONTIER_OUT),
            "--repo-id", HARD_FRONTIER_HF_REPO_ID,
            "--base-model", "Qwen/Qwen3-1.7B",
            "--include-zip",
        ],
        check=True,
    )
    print("HF publish done:", f"https://huggingface.co/{HARD_FRONTIER_HF_REPO_ID}")
elif RUN_HARD_FRONTIER_TRACK:
    print("HF_TOKEN not set, so Track C publish was skipped.")
else:
    print("Skipping Track C publish.")
